# Исследование 

Импорты

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 

Настройки отображения

In [3]:
pd.set_option("display.max_rows", None)

### Сбор и склейка датасетов по нужным признакам

Для исследования были выбраны датасеты следующих стран:
- Босния и Герцеговина
- Болгария
- Венгрия
- Ирландия
- Черногория
- Морокко
- Португалия
- Швеция
- Тунис

Ниже приведен полный процесс создания обработки сырых данных и создания двух датасетов для двух разных HGF таргетов: HGF по Sales и HGF по employees. 

##### Для обучения моделей и анализа будут использованы признаки, представленные в словаре ниже:

In [4]:
FEATURES = {
    # =========================
    # 1. Финансовые признаки
    # =========================
    "d2": "Продажи фирмы за последний завершенный финансовый год",
    "n3": "Продажи фирмы три финансовых года назад",
    "d1a3": "Доля основного продукта или услуги в общей выручке",

    "n2a": "Общие затраты на труд, включая заработную плату и выплаты работникам",
    "n2b": "Расходы на электроэнергию",
    "n2e": "Расходы на сырье и промежуточные материалы",
    "n2i": "Расходы на товары, приобретенные для перепродажи",

    "n5a": "Инвестиции в машины и оборудование",
    "n5b": "Инвестиции в землю и здания",
    "n7a": "Стоимость замещения имеющихся машин и оборудования",

    "k4": "Приобретала ли фирма основные средства",
    "k5a": "Доля инвестиций, профинансированная за счет собственных средств",
    "k5f": "Доля инвестиций, профинансированная за счет кредита поставщиков или авансов клиентов",
    "k5i": "Доля инвестиций, профинансированная за счет вкладов владельцев или нового капитала",

    "k7": "Наличие овердрафта",
    "k9": "Тип финансовой организации, выдавшей кредит",
    "k11": "Размер последнего кредита на момент его одобрения",
    "k13": "Требовалось ли обеспечение или залог для получения кредита",

    "k14a": "Использовались ли земля или здания в качестве залога",
    "k14b": "Использовались ли машины или оборудование в качестве залога",
    "k14c": "Использовалась ли дебиторская задолженность в качестве залога",
    "k14d": "Использовались ли личные активы владельцев в качестве залога",
    "k14e": "Использовался ли другой вид залога",

    "k17": "Причина, по которой фирма не обращалась за кредитом",
    "k21": "Проверялась ли финансовая отчетность внешним аудитором",
    "k30": "Насколько доступ к финансированию является препятствием для деятельности фирмы",

    # ==================================================
    # 2. Структура фирмы, собственность и менеджмент
    # ==================================================
    "a3": "Размер населенного пункта, в котором расположена фирма",
    "a7": "Является ли предприятие частью фирмы с несколькими подразделениями",
    "a9": "Ведется ли отдельная финансовая отчетность предприятия",
    "a11": "Является ли отчетность головного офиса независимой от других подразделений",

    "b1": "Юридическая форма предприятия",
    "b2a": "Доля внутренней частной собственности",
    "b2b": "Доля иностранной частной собственности",
    "b2c": "Доля государственной собственности",
    "b2d": "Доля прочих видов собственности",

    "b4": "Есть ли женщины среди владельцев предприятия",
    "b5": "Год начала деятельности предприятия",
    "b6": "Количество работников при создании предприятия",
    "b7": "Опыт топ-менеджера в данной отрасли",
    "b7a": "Является ли топ-менеджер женщиной",
    "b8": "Наличие международно признанного сертификата качества",

    # ================================================
    # 3. Рынок, экспорт, технологии и инновации
    # ================================================
    "d12a": "Доля материалов и ресурсов внутреннего происхождения",
    "d12b": "Доля импортных материалов и ресурсов",
    "d13": "Импортирует ли фирма материалы напрямую",

    "d3a": "Доля продаж на внутреннем рынке",
    "d3b": "Доля продаж через косвенный экспорт",
    "d3c": "Доля продаж через прямой экспорт",
    "d8": "Год начала экспортной деятельности",

    "e1": "Основной рынок главного продукта или услуги",
    "e11": "Конкурирует ли фирма с предприятиями неформального сектора",
    "e6": "Использует ли фирма технологии, лицензированные у иностранной компании",

    "c22b": "Наличие собственного веб-сайта",
    "h1": "Внедряла ли фирма новые или существенно улучшенные продукты или услуги",

    # =========================================
    # 4. Персонал и человеческий капитал
    # =========================================
    "l1": "Количество постоянных работников в последнем финансовом году",
    "l2": "Количество постоянных работников три финансовых года назад",

    "l3a": "Количество постоянных производственных работников",
    "l3b": "Количество постоянных непроизводственных работников",
    "l4b": "Количество низкоквалифицированных производственных работников",

    "l5": "Количество или доля женщин среди постоянных работников",
    "l5a": "Количество или доля женщин среди производственных работников",
    "l5b": "Количество или доля женщин среди непроизводственных работников",

    "l6": "Количество временных или сезонных работников",
    "l8": "Средняя продолжительность работы временных или сезонных работников",

    "l10": "Проводила ли фирма формальное обучение работников",
    "l11a": "Доля производственных работников, прошедших формальное обучение",
    "l11b": "Доля непроизводственных работников, прошедших формальное обучение",

    # =============================================
    # 5. Государственное регулирование и коррупция
    # =============================================
    "j2": "Доля времени руководства, затрачиваемая на выполнение требований государственных органов",
    "j3": "Проводились ли налоговые проверки предприятия",
    "j4": "Количество или частота налоговых проверок",
    "j5": "Ожидались ли неформальные платежи при налоговых проверках",

    "j6a": "Пыталась ли фирма получить государственный контракт",

    "j10": "Обращалась ли фирма за импортной лицензией",
    "j11": "Время, необходимое для получения импортной лицензии",
    "j12": "Ожидались ли неформальные платежи при получении импортной лицензии",

    "j13": "Обращалась ли фирма за операционной лицензией",
    "j14": "Время, необходимое для получения операционной лицензии",
    "j15": "Ожидались ли неформальные платежи при получении операционной лицензии",

    # ==================================
    # 6. Бизнес-среда и препятствия
    # ==================================
    "d30a": "Насколько транспорт является препятствием для деятельности фирмы",
    "d30b": "Насколько таможенное и торговое регулирование является препятствием",
    "g30a": "Насколько доступ к земле является препятствием",
    "h30": "Насколько судебная система является препятствием",
    "i30": "Насколько преступность, кражи и беспорядки являются препятствием",

    "j30a": "Насколько налоговые ставки являются препятствием",
    "j30c": "Насколько лицензирование и разрешения являются препятствием",
    "j30e": "Насколько политическая нестабильность является препятствием",
    "j30f": "Насколько коррупция является препятствием",

    "l30b": "Насколько недостаточная квалификация рабочей силы является препятствием",
    "m1a": "Главное препятствие для деятельности предприятия"
}

#### Загрузка данных

In [5]:
datasets = {
    'bos': pd.read_stata("Bosnia and Herzegovina _2009_2013_2019_2023_.dta", convert_categoricals=False),
    'bol': pd.read_stata("Bulgaria _2009_2013_2019_2023_.dta", convert_categoricals=False),
    'hun': pd.read_stata("Hungary _2009_2013_2019_2023_.dta", convert_categoricals=False),
    'irl': pd.read_stata("Ireland _2020_2024_.dta", convert_categoricals=False),
    'mon': pd.read_stata("Montenegro _2009_2013_2019_2023_.dta", convert_categoricals=False),
    'mor': pd.read_stata("Morocco-2013-2019-2023.dta", convert_categoricals=False),
    'por': pd.read_stata("Portugal_2019_2023.dta", convert_categoricals=False),
    'swe': pd.read_stata("Sweden _2014_2020_2024_.dta", convert_categoricals=False),
    'tun': pd.read_stata("Tunisia _2013_2020_2024_.dta", convert_categoricals=False)
}

Изучение датасетов:

Проверим, есть ли в датасетах все нужные нам признаки:

In [6]:
feature_cols = list(FEATURES.keys())

target_cols = [
    "d2",
    "n3",
    "l1",
    "l2",
    "a20y"
]

technical_cols = [
    "panelid",
    "year"
]

required_cols = technical_cols + feature_cols + target_cols


for country, df in datasets.items():
    print("=" * 70)
    print(country)
    print("Размер:", df.shape)

    missing = [col for col in required_cols if col not in df.columns]

    print("Найдено признаков:",
          len([col for col in feature_cols if col in df.columns]),
          "/",
          len(feature_cols))

    print("Отсутствует:", missing)

bos
Размер: (1434, 802)
Найдено признаков: 88 / 88
Отсутствует: ['a20y']
bol
Размер: (2063, 805)
Найдено признаков: 88 / 88
Отсутствует: ['a20y']
hun
Размер: (2237, 811)
Найдено признаков: 88 / 88
Отсутствует: ['a20y']
irl
Размер: (1215, 494)
Найдено признаков: 88 / 88
Отсутствует: []
mon
Размер: (567, 811)
Найдено признаков: 88 / 88
Отсутствует: ['a20y']
mor
Размер: (2101, 837)
Найдено признаков: 88 / 88
Отсутствует: ['a20y']
por
Размер: (2069, 559)
Найдено признаков: 88 / 88
Отсутствует: []
swe
Размер: (1791, 630)
Найдено признаков: 88 / 88
Отсутствует: ['a20y']
tun
Размер: (1852, 738)
Найдено признаков: 88 / 88
Отсутствует: ['a20y']


###### обработка a20y

In [7]:
for country, df in datasets.items():
    cols = [
        col for col in df.columns
        if "a20y" in col.lower()
    ]
    print(country, cols)

bos ['_2019_2023_a20y']
bol ['_2019_2023_a20y']
hun ['_2019_2023_a20y']
irl ['a20y']
mon ['_2019_2023_a20y']
mor ['_2019_2023_a20y']
por ['a20y']
swe ['_2020_2024_a20y', '_2014_a20ystart', '_2014_a20yend']
tun ['_2020_2024_a20y']


In [8]:
A20Y_COLUMNS = {
    "bos": "_2019_2023_a20y",
    "bol": "_2019_2023_a20y",
    "hun": "_2019_2023_a20y",
    "irl": "a20y",
    "mon": "_2019_2023_a20y",
    "mor": "_2019_2023_a20y",
    "por": "a20y",
    "swe": "_2020_2024_a20y",
    "tun": "_2020_2024_a20y",
}

In [9]:
for name, df in datasets.items():
    old_name = A20Y_COLUMNS[name]

    if old_name != "a20y":
        df.rename(columns={old_name: "a20y"}, inplace=True)

In [10]:
for country, df in datasets.items():
    cols = [
        col for col in df.columns
        if "a20y" in col.lower()
    ]
    print(country, cols)

bos ['a20y']
bol ['a20y']
hun ['a20y']
irl ['a20y']
mon ['a20y']
mor ['a20y']
por ['a20y']
swe ['a20y', '_2014_a20ystart', '_2014_a20yend']
tun ['a20y']


#### Создание единого датасета

Определение и проверки для baseline и target годов

In [11]:
PANEL_YEARS = {
    "bos": (2019, 2023),
    "bol": (2019, 2023),
    "hun": (2019, 2023),
    "mon": (2019, 2023),
    "mor": (2019, 2023),
    "por": (2019, 2023),

    "irl": (2020, 2024),
    "swe": (2020, 2024),
    "tun": (2020, 2024),
}

In [12]:
for name, df in datasets.items():
    print(name, sorted(df["year"].dropna().unique()))

bos [np.float32(2009.0), np.float32(2013.0), np.float32(2019.0), np.float32(2023.0)]
bol [np.float32(2009.0), np.float32(2013.0), np.float32(2019.0), np.float32(2023.0)]
hun [np.float32(2009.0), np.float32(2013.0), np.float32(2019.0), np.float32(2023.0)]
irl [np.float32(2020.0), np.float32(2024.0)]
mon [np.float32(2009.0), np.float32(2013.0), np.float32(2019.0), np.float32(2023.0)]
mor [np.float32(2013.0), np.float32(2019.0), np.float32(2023.0)]
por [np.float32(2019.0), np.float32(2023.0)]
swe [np.float32(2014.0), np.float32(2020.0), np.float32(2024.0)]
tun [np.float32(2013.0), np.float32(2020.0), np.float32(2024.0)]


In [13]:
panels = {}

for name, df in datasets.items():
    baseline_year, target_year = PANEL_YEARS[name]

    baseline = df[df["year"] == baseline_year].copy()
    target = df[df["year"] == target_year].copy()

    print(
        name,
        "baseline:", len(baseline),
        "target:", len(target)
    )

    panels[name] = {
        "baseline": baseline,
        "target": target
    }

bos baseline: 362 target: 351
bol baseline: 772 target: 710
hun baseline: 805 target: 831
irl baseline: 606 target: 609
mon baseline: 150 target: 151
mor baseline: 1096 target: 598
por baseline: 1062 target: 1007
swe baseline: 591 target: 600
tun baseline: 615 target: 645


Проверка панелей на присутствие в начале и конце опроса:

In [14]:
for name, p in panels.items():
    baseline = p["baseline"]
    target = p["target"]

    common_ids = set(baseline["panelid"]) & set(target["panelid"])

    print(
        name,
        "baseline:", len(baseline),
        "target:", len(target),
        "common:", len(common_ids)
    )

bos baseline: 362 target: 351 common: 131
bol baseline: 772 target: 710 common: 218
hun baseline: 805 target: 831 common: 295
irl baseline: 606 target: 609 common: 190
mon baseline: 150 target: 151 common: 87
mor baseline: 1096 target: 598 common: 280
por baseline: 1062 target: 1007 common: 308
swe baseline: 591 target: 600 common: 180
tun baseline: 615 target: 645 common: 105


Проверка на дубликаты

In [15]:
for name, p in panels.items():
    for period in ["baseline", "target"]:
        df = p[period]

        print(
            name,
            period,
            "duplicates:",
            df["panelid"].duplicated().sum()
        )

bos baseline duplicates: 0
bos target duplicates: 0
bol baseline duplicates: 0
bol target duplicates: 0
hun baseline duplicates: 0
hun target duplicates: 0
irl baseline duplicates: 0
irl target duplicates: 0
mon baseline duplicates: 0
mon target duplicates: 0
mor baseline duplicates: 0
mor target duplicates: 0
por baseline duplicates: 0
por target duplicates: 0
swe baseline duplicates: 0
swe target duplicates: 0
tun baseline duplicates: 0
tun target duplicates: 0


In [16]:
for name, p in panels.items():
    print(name)
    print(p["target"]["a20y"].value_counts(dropna=False))
    print()

bos
a20y
2022.0    351
Name: count, dtype: int64

bol
a20y
2022.0    710
Name: count, dtype: int64

hun
a20y
2022.0    831
Name: count, dtype: int64

irl
a20y
2023    418
2024    179
2022     12
Name: count, dtype: int64

mon
a20y
2022.0    150
2023.0      1
Name: count, dtype: int64

mor
a20y
2022.0    535
2023.0     63
Name: count, dtype: int64

por
a20y
2022    1000
2023       7
Name: count, dtype: int64

swe
a20y
2023.0    506
2024.0     61
2022.0     33
Name: count, dtype: int64

tun
a20y
2023.0    554
2024.0     91
Name: count, dtype: int64



In [17]:
FEATURE_TIMING = {
    "d2": "last_fiscal_year",
    "n3": "three_fiscal_years_ago",
    "b7": "survey_time",
    "c22b": "survey_time",
    "d3c": "last_fiscal_year",
    "h1": "previous_3_years",
    "l10": "last_fiscal_year",
}

In [18]:
EXPECTED_A20Y = {
    "bos": 2022,
    "bol": 2022,
    "hun": 2022,
    "mon": 2022,
    "mor": 2022,
    "por": 2022,

    "irl": 2023,
    "swe": 2023,
    "tun": 2023,
}

In [19]:
feature_cols = list(FEATURES.keys())

In [20]:
target_cols = [
    "panelid",
    "a20y",
    "d2",
    "n3",
    "l1",
    "l2"
]

In [21]:
prepared = {}

for name, p in panels.items():

    baseline = p["baseline"].copy()
    target = p["target"].copy()

    # Оставляем правильный финансовый период
    target = target[
        target["a20y"] == EXPECTED_A20Y[name]
    ].copy()

    # Признаки исходной волны
    baseline = baseline[
        ["panelid"] + feature_cols
    ].copy()

    # Данные будущей волны
    target = target[target_cols].copy()

    target = target.rename(columns={
        "d2": "sales_end",
        "n3": "sales_start",
        "l1": "employees_end",
        "l2": "employees_start",
        "a20y": "financial_year_end"
    })

    # Оставляем только фирмы, присутствующие в обеих волнах
    df = baseline.merge(
        target,
        on="panelid",
        how="inner"
    )

    df["country"] = name

    prepared[name] = df

    print(name, len(df))

bos 131
bol 218
hun 295
irl 148
mon 87
mor 252
por 305
swe 161
tun 103


Проверка заполненности колонок для target переменных

In [22]:
for name, df in prepared.items():

    print("=" * 60)
    print(name)

    print("Всего:", len(df))

    print(
        "sales_start заполнено:",
        df["sales_start"].notna().sum()
    )

    print(
        "sales_end заполнено:",
        df["sales_end"].notna().sum()
    )

    print(
        "employees_start заполнено:",
        df["employees_start"].notna().sum()
    )

    print(
        "sales_start > 0:",
        (df["sales_start"] > 0).sum()
    )

    print(
        "sales_end > 0:",
        (df["sales_end"] > 0).sum()
    )

    print(
        "employees_start >= 10:",
        (df["employees_start"] >= 10).sum()
    )

bos
Всего: 131
sales_start заполнено: 131
sales_end заполнено: 131
employees_start заполнено: 131
sales_start > 0: 127
sales_end > 0: 130
employees_start >= 10: 104
bol
Всего: 218
sales_start заполнено: 218
sales_end заполнено: 218
employees_start заполнено: 218
sales_start > 0: 213
sales_end > 0: 218
employees_start >= 10: 176
hun
Всего: 295
sales_start заполнено: 295
sales_end заполнено: 295
employees_start заполнено: 295
sales_start > 0: 282
sales_end > 0: 294
employees_start >= 10: 209
irl
Всего: 148
sales_start заполнено: 148
sales_end заполнено: 148
employees_start заполнено: 148
sales_start > 0: 146
sales_end > 0: 146
employees_start >= 10: 81
mon
Всего: 87
sales_start заполнено: 87
sales_end заполнено: 87
employees_start заполнено: 87
sales_start > 0: 82
sales_end > 0: 87
employees_start >= 10: 57
mor
Всего: 252
sales_start заполнено: 252
sales_end заполнено: 252
employees_start заполнено: 252
sales_start > 0: 249
sales_end > 0: 252
employees_start >= 10: 216
por
Всего: 305
sal

Отбор только ИНФОРМАТИВНЫХ (больших 0) значений таргетов

In [23]:
for name, df in prepared.items():
    df["sales_start"] = df["sales_start"].mask(df["sales_start"] < 0, np.nan)
    df["sales_end"] = df["sales_end"].mask(df["sales_end"] < 0, np.nan)

In [24]:
valid = (
    df["sales_start"].notna()
    & df["sales_end"].notna()
    & (df["sales_start"] > 0)
    & (df["employees_start"] >= 10)
)

In [25]:
sales_samples = {}

for name, df in prepared.items():
    df = df.copy()

    # Оставляем только фирмы, для которых можно корректно определить HGF
    valid = (
        df["sales_start"].notna()
        & df["sales_end"].notna()
        & df["employees_start"].notna()
        & (df["sales_start"] > 0)
        & (df["sales_end"] >= 0)
        & (df["employees_start"] >= 10)
    )

    df = df.loc[valid].copy()

    # Среднегодовой темп роста продаж за 3 года
    df["sales_cagr"] = (
        (df["sales_end"] / df["sales_start"]) ** (1 / 3) - 1
    )

    # HGF по продажам
    df["hgf_sales"] = (df["sales_cagr"] > 0.10).astype(int)

    sales_samples[name] = df

    print(
        name,
        "firms:", len(df),
        "HGF:", df["hgf_sales"].sum(),
        "HGF share:", round(df["hgf_sales"].mean() * 100, 1), "%"
    )

bos firms: 101 HGF: 41 HGF share: 40.6 %
bol firms: 172 HGF: 55 HGF share: 32.0 %
hun firms: 200 HGF: 112 HGF share: 56.0 %
irl firms: 80 HGF: 26 HGF share: 32.5 %
mon firms: 52 HGF: 29 HGF share: 55.8 %
mor firms: 216 HGF: 32 HGF share: 14.8 %
por firms: 231 HGF: 114 HGF share: 49.4 %
swe firms: 124 HGF: 27 HGF share: 21.8 %
tun firms: 64 HGF: 25 HGF share: 39.1 %


Финальное объединение датасетов в конечный:


In [26]:
master_df = pd.concat(
    sales_samples.values(),
    ignore_index=True
)

In [27]:
print("Всего фирм:", len(master_df))
print("HGF:", master_df["hgf_sales"].sum())
print("Non-HGF:", (master_df["hgf_sales"] == 0).sum())
print("Доля HGF:", master_df["hgf_sales"].mean())

Всего фирм: 1240
HGF: 461
Non-HGF: 779
Доля HGF: 0.3717741935483871


In [28]:
print(master_df.columns)

Index(['panelid', 'd2', 'n3', 'd1a3', 'n2a', 'n2b', 'n2e', 'n2i', 'n5a', 'n5b',
       'n7a', 'k4', 'k5a', 'k5f', 'k5i', 'k7', 'k9', 'k11', 'k13', 'k14a',
       'k14b', 'k14c', 'k14d', 'k14e', 'k17', 'k21', 'k30', 'a3', 'a7', 'a9',
       'a11', 'b1', 'b2a', 'b2b', 'b2c', 'b2d', 'b4', 'b5', 'b6', 'b7', 'b7a',
       'b8', 'd12a', 'd12b', 'd13', 'd3a', 'd3b', 'd3c', 'd8', 'e1', 'e11',
       'e6', 'c22b', 'h1', 'l1', 'l2', 'l3a', 'l3b', 'l4b', 'l5', 'l5a', 'l5b',
       'l6', 'l8', 'l10', 'l11a', 'l11b', 'j2', 'j3', 'j4', 'j5', 'j6a', 'j10',
       'j11', 'j12', 'j13', 'j14', 'j15', 'd30a', 'd30b', 'g30a', 'h30', 'i30',
       'j30a', 'j30c', 'j30e', 'j30f', 'l30b', 'm1a', 'financial_year_end',
       'sales_end', 'sales_start', 'employees_end', 'employees_start',
       'country', 'sales_cagr', 'hgf_sales'],
      dtype='str')


Таким образом данный датасет является исходным для HGF по Sales для начала работы с машинным обучением 

In [29]:
master_sales = master_df.copy()

Построим датасет для определения HGF по employees

In [30]:
employee_samples = {}

for name, df in prepared.items():
    df = df.copy()

    # Служебные отрицательные значения WBES -> NaN
    df["employees_start"] = df["employees_start"].mask(
        df["employees_start"] < 0,
        np.nan
    )

    df["employees_end"] = df["employees_end"].mask(
        df["employees_end"] < 0,
        np.nan
    )

    # Условия для определения HGF по занятости
    valid = (
        df["employees_start"].notna()
        & df["employees_end"].notna()
        & (df["employees_start"] >= 10)
        & (df["employees_end"] >= 0)
    )

    df = df.loc[valid].copy()

    # Среднегодовой рост занятости за 3 года
    df["employees_cagr"] = (
        (df["employees_end"] / df["employees_start"]) ** (1 / 3) - 1
    )

    # HGF по занятости
    df["hgf_employees"] = (
        df["employees_cagr"] > 0.10
    ).astype(int)

    employee_samples[name] = df

    print(
        name,
        "firms:", len(df),
        "HGF:", df["hgf_employees"].sum(),
        "HGF share:",
        round(df["hgf_employees"].mean() * 100, 1),
        "%"
    )

bos firms: 104 HGF: 6 HGF share: 5.8 %
bol firms: 176 HGF: 9 HGF share: 5.1 %
hun firms: 209 HGF: 16 HGF share: 7.7 %
irl firms: 81 HGF: 11 HGF share: 13.6 %
mon firms: 57 HGF: 3 HGF share: 5.3 %
mor firms: 216 HGF: 35 HGF share: 16.2 %
por firms: 231 HGF: 18 HGF share: 7.8 %
swe firms: 128 HGF: 7 HGF share: 5.5 %
tun firms: 71 HGF: 3 HGF share: 4.2 %


In [31]:
master_employees = pd.concat(
    employee_samples.values(),
    ignore_index=True
)

In [32]:
print("Всего фирм:", len(master_employees))
print("HGF:", master_employees["hgf_employees"].sum())
print(
    "Non-HGF:",
    (master_employees["hgf_employees"] == 0).sum()
)
print(
    "Доля HGF:",
    master_employees["hgf_employees"].mean()
)

Всего фирм: 1273
HGF: 108
Non-HGF: 1165
Доля HGF: 0.08483896307934014


### Обработка признаков

##### Обработка null значений

Для начала посмотрим на nan values по каждой колонке в датасетах

In [33]:
print("NaN values in master_sales:")
master_sales.isna().sum().sort_values(ascending=False)

NaN values in master_sales:


a9                    1185
j15                   1141
j14                   1141
j11                   1105
j12                   1105
a11                   1072
n2i                   1051
l11a                  1014
l11b                  1014
k14c                   875
k14b                   875
k14a                   875
k14d                   875
k14e                   875
l8                     818
l5                     771
j4                     727
j5                     727
k5i                    667
k5a                    667
k5f                    667
k11                    639
k13                    639
k9                     639
d8                     633
n5a                    576
n5b                    576
l3a                    470
l4b                    470
l3b                    470
l5a                    469
n7a                    469
n2e                    469
l5b                    469
d13                    437
k17                    377
b1                       0
k

In [34]:
print("NaN values in master_employees:")
master_employees.isna().sum().sort_values(ascending=False)

NaN values in master_employees:


a9                    1214
j15                   1170
j14                   1170
j11                   1131
j12                   1131
a11                   1102
n2i                   1079
l11a                  1040
l11b                  1040
k14c                   894
k14b                   894
k14a                   894
k14d                   894
k14e                   894
l8                     835
l5                     789
j4                     745
j5                     745
k5i                    681
k5a                    681
k5f                    681
k11                    651
k13                    651
k9                     651
d8                     648
n5a                    587
n5b                    587
l3a                    485
l4b                    485
l3b                    485
l5a                    484
n7a                    484
n2e                    484
l5b                    484
d13                    442
k17                    392
sales_start             31
s

Пропуски вызваны как просто отсутствием данных, так и связаны с различными причинами и ответами опрошенных компаний, поэтому была выбрана следующая стратегия обработки признаков:

- Признаки с >70% пропусков будут удалены
- Остальные признаки будут заполнены медианой + к ним будет добавлен столбец с данными о причине отсутствия данных

Удаление признаков:

In [35]:
threshold = 0.70

features = list(FEATURES)

drop_cols = set(
    master_sales[features].columns[
        master_sales[features].isna().mean() > threshold
    ]
) | set(
    master_employees[features].columns[
        master_employees[features].isna().mean() > threshold
    ]
)

master_sales = master_sales.drop(columns=drop_cols)
master_employees = master_employees.drop(columns=drop_cols)

print("Удалено признаков:", len(drop_cols))
print("Удаленные признаки:", sorted(drop_cols))

Удалено признаков: 14
Удаленные признаки: ['a11', 'a9', 'j11', 'j12', 'j14', 'j15', 'k14a', 'k14b', 'k14c', 'k14d', 'k14e', 'l11a', 'l11b', 'n2i']


Среди данных встречаются следующие "отрицательные" коды требующие обработки

In [36]:
for name, df in {
    "master_sales": master_sales,
    "master_employees": master_employees
}.items():

    print(f"\n{name}")

    features = [col for col in FEATURES if col in df.columns]

    for value in range(-9, 0):
        count = (df[features] == value).sum().sum()

        if count:
            print(value, count)


master_sales
-9 2907
-8 125
-7 451
-6 8

master_employees
-9 2985
-8 129
-7 460
-6 8


Коды "-9", "-8", "-7" имеют одинаковое значение независимо от контекста конкретного признака. Для категории "-6" все же стоит изучить контекст, в котором он встречается, так как он, в свою очередь, может иметь разное значение в зависимости от выбранного признака.

In [37]:
for name, df in {
    "master_sales": master_sales,
    "master_employees": master_employees
}.items():

    print(f"\n{name}")

    features = [col for col in FEATURES if col in df.columns]

    for col in features:
        count = (df[col] == -6).sum()

        if count > 0:
            print(col, count)


master_sales
b8 8

master_employees
b8 8


Код "-6" встречается только в признаке b8. Тут он означает категорию "Still in progress", на которую и будет заменен

Для дальнейшей обработки необходимо разделить признаки на категориальные, бинарные, числовые и порядковые

In [38]:
NUMERIC_FEATURES = [
    "d2", "n3", "d1a3",
    "n2a", "n2b", "n2e",
    "n5a", "n5b", "n7a",
    "k5a", "k5f", "k5i", "k11",

    "b2a", "b2b", "b2c", "b2d",
    "b5", "b6", "b7",

    "d12a", "d12b",
    "d3a", "d3b", "d3c",
    "d8",

    "l1", "l2",
    "l3a", "l3b", "l4b",
    "l5", "l5a", "l5b",
    "l6", "l8",

    "j2", "j4"
]

In [39]:
BINARY_FEATURES = [
    "k4",
    "k7",
    "k13",
    "k21",

    "a7",

    "b4",
    "b7a",
    "b8",

    "d13",
    "e11",
    "e6",
    "c22b",
    "h1",

    "l10",

    "j3",
    "j5",
    "j6a",
    "j10",
    "j13"
]

In [40]:
CATEGORICAL_FEATURES = [
    "k9",
    "k17",

    "b1",

    "e1",

    "m1a"
]

In [41]:
ORDINAL_FEATURES = [
    "k30",
    "a3",

    "d30a",
    "d30b",
    "g30a",
    "h30",
    "i30",
    "j30a",
    "j30c",
    "j30e",
    "j30f",
    "l30b"
]

In [42]:
CAT_FEATURES = (
    BINARY_FEATURES
    + CATEGORICAL_FEATURES
    + ORDINAL_FEATURES
)

Далее для качественной обработки пропусков создадим столбцы status для каждой переменной, чтобы зафиксифровать причины пропуска того или иного значения

In [43]:
def process_missing_status(df):
    df = df.copy()

    # Числовые признаки
    for col in NUMERIC_FEATURES:

        mask = (
            df[col].isna()
            | df[col].isin([-7, -8, -9])
        )

        # Если проблемных значений нет, status не создаем
        if not mask.any():
            continue

        status_col = f"{col}_status"

        df[status_col] = "observed"

        df.loc[df[col] == -7, status_col] = "not_applicable"
        df.loc[df[col] == -8, status_col] = "refused"
        df.loc[df[col] == -9, status_col] = "dont_know"
        df.loc[df[col].isna(), status_col] = "missing"

        # После сохранения причины специальные коды превращаем в NaN
        df[col] = df[col].replace([-7, -8, -9], np.nan)

    # Категориальные признаки
    for col in CAT_FEATURES:
        df[col] = df[col].astype("object")

        df[col] = df[col].replace({
            -7: "not_applicable",
            -8: "refused",
            -9: "dont_know"
        })

        df[col] = df[col].fillna("missing")

    # Особый случай b8
    df["b8"] = df["b8"].replace({
        -6: "still_in_process"
    })

    return df

In [44]:
master_sales = process_missing_status(master_sales)
master_employees = process_missing_status(master_employees)

По завершении обработки было создано следующее количество новых столбцов:

In [45]:
status_cols_sales = [
    col for col in master_sales.columns
    if col.endswith("_status")
]

status_cols_employees = [
    col for col in master_employees.columns
    if col.endswith("_status")
]

print("Sales status columns:", len(status_cols_sales))
print("Employees status columns:", len(status_cols_employees))

Sales status columns: 38
Employees status columns: 38


Добавим новые столбцы к категориальным 

In [46]:
STATUS_FEATURES = [
    col for col in master_sales.columns
    if col.endswith("_status")
]

MODEL_CAT_FEATURES = CAT_FEATURES + STATUS_FEATURES + ["country"]
MODEL_FEATURES = NUMERIC_FEATURES + MODEL_CAT_FEATURES

Следующим шагом является импутация значений. Перед ней необходимо разделить данные на X и y, а так же train/test выборки, чтобы избежать ненужных утечек данных

In [47]:
X_sales = master_sales[MODEL_FEATURES].copy()
y_sales = master_sales["hgf_sales"].copy()

In [48]:
X_employees = master_employees[MODEL_FEATURES].copy()
y_employees = master_employees["hgf_employees"].copy()

In [49]:
from sklearn.model_selection import train_test_split

X_train_sales, X_test_sales, y_train_sales, y_test_sales = train_test_split(
    X_sales,
    y_sales,
    test_size=0.2,
    stratify=y_sales,
    random_state=42
)

X_train_emp, X_test_emp, y_train_emp, y_test_emp = train_test_split(
    X_employees,
    y_employees,
    test_size=0.2,
    stratify=y_employees,
    random_state=42
)

In [50]:
print("Sales:")
print(len(X_train_sales), len(X_test_sales))
print(y_train_sales.value_counts())
print(y_test_sales.value_counts())

print("\nEmployees:")
print(len(X_train_emp), len(X_test_emp))
print(y_train_emp.value_counts())
print(y_test_emp.value_counts())

Sales:
992 248
hgf_sales
0    623
1    369
Name: count, dtype: int64
hgf_sales
0    156
1     92
Name: count, dtype: int64

Employees:
1018 255
hgf_employees
0    932
1     86
Name: count, dtype: int64
hgf_employees
0    233
1     22
Name: count, dtype: int64


После успешного разделения можно приступить к заполнению датасетов

In [51]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

X_train_sales[NUMERIC_FEATURES] = imputer.fit_transform(
    X_train_sales[NUMERIC_FEATURES]
)

X_test_sales[NUMERIC_FEATURES] = imputer.transform(
    X_test_sales[NUMERIC_FEATURES]
)

In [52]:
imputer_emp = SimpleImputer(strategy="median")

X_train_emp[NUMERIC_FEATURES] = imputer_emp.fit_transform(
    X_train_emp[NUMERIC_FEATURES]
)

X_test_emp[NUMERIC_FEATURES] = imputer_emp.transform(
    X_test_emp[NUMERIC_FEATURES]
)

##### Выбор моделей

Выбирая baseline модель и модели для дальнейшего сравнения, было решено остановиться на следующих вариантах:

Для baseline модели использовать классический LogisticClassifier

Для дальнейшего исследования будут использоваться RandomForestClassifier, LightGBM, HistGradientBoost, SVC, KNN

Выбор обоснован различной структурой моделей

##### Кодировка признаков

Чтобы обучить данные модели необходимо закодировать категориальные признаки, а также нормализовать числовые. Нормализация числовых признаков нужна только для логистической и SVC модели, и для этих же моделей будут созданы два датасета с one-hot encoding, в то время как нормализация для остальных моделей использоваться не будет, а категории будут закодированы при помощи ordinal encoder.

Для начала, все категории должны быть приведены к строчному типу:

In [55]:
for df in [
    X_train_sales,
    X_test_sales,
    X_train_emp,
    X_test_emp
]:
    df[MODEL_CAT_FEATURES] = df[MODEL_CAT_FEATURES].astype(str)

После этого, можно начинать непосредственную кодировку значений

Произведем one-hot encoding категориальных признаков

In [56]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

ohe_sales = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            MODEL_CAT_FEATURES
        )
    ],
    remainder="passthrough",
    verbose_feature_names_out=False
)

X_train_sales_ohe = ohe_sales.fit_transform(X_train_sales)
X_test_sales_ohe = ohe_sales.transform(X_test_sales)

ohe_sales.set_output(transform="pandas")

X_train_sales_ohe = ohe_sales.transform(X_train_sales)
X_test_sales_ohe = ohe_sales.transform(X_test_sales)

In [57]:
ohe_emp = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            MODEL_CAT_FEATURES
        )
    ],
    remainder="passthrough",
    verbose_feature_names_out=False
)

ohe_emp.fit(X_train_emp)
ohe_emp.set_output(transform="pandas")

X_train_emp_ohe = ohe_emp.transform(X_train_emp)
X_test_emp_ohe = ohe_emp.transform(X_test_emp)

Далее следует обычная кодировка категорий при помощи ordinal encoding

In [58]:
from sklearn.preprocessing import OrdinalEncoder

ordinal_sales = ColumnTransformer(
    transformers=[
        (
            "cat",
            OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1
            ),
            MODEL_CAT_FEATURES
        )
    ],
    remainder="passthrough",
    verbose_feature_names_out=False
)

ordinal_sales.fit(X_train_sales)
ordinal_sales.set_output(transform="pandas")

X_train_sales_ord = ordinal_sales.transform(X_train_sales)
X_test_sales_ord = ordinal_sales.transform(X_test_sales)

In [59]:
ordinal_emp = ColumnTransformer(
    transformers=[
        (
            "cat",
            OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1
            ),
            MODEL_CAT_FEATURES
        )
    ],
    remainder="passthrough",
    verbose_feature_names_out=False
)

ordinal_emp.fit(X_train_emp)
ordinal_emp.set_output(transform="pandas")

X_train_emp_ord = ordinal_emp.transform(X_train_emp)
X_test_emp_ord = ordinal_emp.transform(X_test_emp)

Robust scaling:

In [60]:
from sklearn.preprocessing import RobustScaler

# =========================
# SALES
# =========================

X_train_sales_linear = X_train_sales_ohe.copy()
X_test_sales_linear = X_test_sales_ohe.copy()

scaler_sales = RobustScaler()

X_train_sales_linear[NUMERIC_FEATURES] = scaler_sales.fit_transform(
    X_train_sales_linear[NUMERIC_FEATURES]
)

X_test_sales_linear[NUMERIC_FEATURES] = scaler_sales.transform(
    X_test_sales_linear[NUMERIC_FEATURES]
)


# =========================
# EMPLOYEES
# =========================

X_train_employees_linear = X_train_emp_ohe.copy()
X_test_employees_linear = X_test_emp_ohe.copy()

scaler_employees = RobustScaler()

X_train_employees_linear[NUMERIC_FEATURES] = scaler_employees.fit_transform(
    X_train_employees_linear[NUMERIC_FEATURES]
)

X_test_employees_linear[NUMERIC_FEATURES] = scaler_employees.transform(
    X_test_employees_linear[NUMERIC_FEATURES]
)

In [61]:
X_train_sales_tree = X_train_sales_ord.copy()
X_test_sales_tree = X_test_sales_ord.copy()

X_train_employees_tree = X_train_emp_ord.copy()
X_test_employees_tree = X_test_emp_ord.copy()

Таким образом были получены следующие датасеты для обучения моделей:

Для SALES HGF

Linear / SVC:
- X_train_sales_linear
- X_test_sales_linear
- y_train_sales
- y_test_sales

Trees:
- X_train_sales_tree
- X_test_sales_tree
- y_train_sales
- y_test_sales


Для EMPLOYEES HGF

Linear / SVC:
- X_train_employees_linear
- X_test_employees_linear
- y_train_employees
- y_test_employees

Trees:
- X_train_employees_tree
- X_test_employees_tree
- y_train_employees
- y_test_employees

### Обучение моделей

Для подбора оптимальных параметров моделей будет использоваться библиотека optuna.

Оптимизируемой метрикой послужит MCC, так как сочитает в себе TP, TN, FP, FN предсказания. Это хорошо подходит для оптимизации всех моделей и позволяет отслеживать реальную производительность и точность модели

In [66]:
import lightgbm
print("LightGBM:", lightgbm.__version__)

ModuleNotFoundError: No module named 'lightgbm'

In [65]:
import optuna
print("Optuna:", optuna.__version__)

Optuna: 4.9.0
